In [1]:
# 1. Installations and Dependencies

# Core Agents & AI
# %pip install -qU langchain-openai
# %pip install -qU langchain-community
# %pip install -qU langgraph
# %pip install -qU python-dotenv
# %pip install -qU tavily-python

# Web Scraping Tools
# %pip install -qU ddgs
# %pip install -qU selenium
# %pip install -qU webdriver-manager
# %pip install -qU beautifulsoup4

#Memory and data persistence

# %pip install -qU aiosqlite
# %pip install -qU langgraph-checkpoint-sqlite

print("[✅] Dependencies configuration checked.")

[✅] Dependencies configuration checked.


In [2]:
# 2. Environment setup: Local LLM (Llama 3 Power)
from langchain_openai import ChatOpenAI

# LM Studio Configuration
lm_studio_base = "http://localhost:1234/v1"
lm_studio_key = "lm-studio" 

# Initialize the LLM
# DICA: Certifique-se que o "Context Length" no LM Studio está setado para 8192 ou mais!
llm = ChatOpenAI(
    model="meta-llama-3.1-8b-instruct", # Atualizado para o Llama 3
    base_url=lm_studio_base,
    api_key=lm_studio_key,
    temperature=0
)

print(f"Target Model: meta-llama-3.1-8b-instruct at {lm_studio_base}")
try:
    response = llm.invoke("System check. Reply 'Online'.").content
    print(f"[✅] Local LLM Status: {response}")
except Exception as e:
    print(f"[❌] Local LLM Connection failed: {e}")

Target Model: meta-llama-3.1-8b-instruct at http://localhost:1234/v1
[✅] Local LLM Status: Online.


In [3]:
# 3. Environment setup: Tavily Search Client
import os
from dotenv import load_dotenv
from tavily import TavilyClient

load_dotenv()

# Verify API Key
tavily_api_key = os.getenv("TAVILY_API_KEY")
if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY not found in .env file.")

# Initialize Client
tavily_client = TavilyClient(tavily_api_key)

print("[✅] Tavily Client initialized.")

# Optional: Quick Connectivity Test
# try:
#     test_response = tavily_client.search(query="test connectivity", max_results=1)
#     if test_response and 'results' in test_response:
#         print("[✅] Tavily API connection successful.")
#     else:
#         print("[⚠️] Tavily connected but returned no results.")
# except Exception as e:
#     print(f"[❌] Tavily Connection failed: {e}")

[✅] Tavily Client initialized.


In [4]:
# 4. Persistence Setup (SQLite Checkpointer)
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# Connect to a local SQLite database to store conversation history
# check_same_thread=False is required for SQLite in this context
db_connection = sqlite3.connect("langgraph_memory.db", check_same_thread=False)

# The 'memory' object is the checkpointer that the graph will use
# It saves the state of the conversation after each step
memory = SqliteSaver(conn=db_connection)

print("[✅] SQLite database connection and checkpointer ready.")

[✅] SQLite database connection and checkpointer ready.


In [5]:
# 5. Config Agent (Manual Force Tool Calling for LM Studio)

from typing import TypedDict, Annotated, List, Dict, Any, Optional
import operator
from langgraph.graph import StateGraph, END
from langchain_core.messages import (
    AnyMessage, SystemMessage, HumanMessage, ToolMessage, BaseMessage
)


class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], operator.add]

class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        self.tools = {t.name: t for t in tools}
        
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)
        graph.add_node("action", self.take_action)
        
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        
        self.graph = graph.compile(checkpointer=checkpointer)
        
        # Ligamos as ferramentas ao modelo. SEM tool_choice aqui.
        self.model = model.bind_tools(tools) 

    def exists_action(self, state: AgentState) -> bool:
        result = state['messages'][-1]
        return hasattr(result, 'tool_calls') and len(result.tool_calls) > 0

    def call_llm(self, state: AgentState) -> Dict[str, List[AnyMessage]]:
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        
        print(f"[🧠] Sending {len(messages)} messages to LLM...")

        # --- NOVA LÓGICA ANTI-LOOP (MANUAL) ---
        # Se for a primeira pergunta, injetamos o 'tool_choice' manualmente
        if len(state['messages']) == 1:
            print("   ↳ First turn. Forcing tool call via 'tool_choice=required'.")
            # Passamos o argumento 'tool_choice' diretamente no invoke.
            # Isso é compatível com o LM Studio.
            message = self.model.invoke(messages, tool_choice="required")
        else:
            # Em turnos subsequentes, o modelo decide (auto)
            message = self.model.invoke(messages)
            
        return {'messages': [message]}
    
    def take_action(self, state: AgentState) -> Dict[str, List[ToolMessage]]:
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"[🛠️] Calling tool: {t['name']} with args: {t['args']}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("[✅] Returning to LLM with tool results.")
        return {'messages': results}

print("[✅] Agent class with 'Manual Force Tool Calling' is ready.")

[✅] Agent class with 'Manual Force Tool Calling' is ready.


In [6]:
# 6. Test 1 - Agent Initialization and First Run (Simplified Tool)

from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage
from datetime import datetime
from langchain_core.tools import tool

# --- 1. Tool Simplification ---
# Criamos uma função simples que só aceita a 'query'.
# Isso evita que o LLM alucine outros parâmetros como 'include_domains'.
@tool
def tavily_search(query: str) -> str:
    """
    A simple search engine tool. Use this to find information on any topic.
    The input should be a search query string.
    """
    # Usamos a ferramenta original "por baixo dos panos"
    search_tool = TavilySearch(max_results=3)
    return search_tool.invoke(query)

print("[✅] Simplified 'tavily_search' tool created.")

# --- 2. System Prompt ---
current_date = datetime.now().strftime("%Y-%m-%d")
SYSTEM_PROMPT = f"""
You are a helpful research assistant. Today's date is {current_date}.
1. To answer the user's question, you MUST first use the 'tavily_search' tool.
2. After receiving the search results, provide a final, comprehensive answer.
""".strip()
print(f"[✅] System prompt defined.")

# --- 3. Agent Initialization ---
# Agora passamos a nossa NOVA ferramenta simplificada para o agente.
abot = Agent(
    llm,
    [tavily_search], # ### MUDANÇA ###
    system=SYSTEM_PROMPT,
    checkpointer=memory
)
print("[✅] Agent initialized with simplified tool.")

# --- 4. Execution ---
thread_config = {"configurable": {"thread_id": "conversation-4"}} # Novo ID

messages = [HumanMessage(content="What is the current weather in São Paulo, Brazil?")]

print("\n" + "="*70)
print(f"🚀 THREAD: {thread_config['configurable']['thread_id']}")
print("="*70)

for event in abot.graph.stream({"messages": messages}, thread_config):
    for node_name, value in event.items():
        print(f"--- Node '{node_name}' ---")
        last_message = value['messages'][-1]
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print(f"  ↳ Planning to call tool: {last_message.tool_calls[0]['name']}")
        else:
            print(last_message.content)

[✅] Simplified 'tavily_search' tool created.
[✅] System prompt defined.
[✅] Agent initialized with simplified tool.

🚀 THREAD: conversation-4
[🧠] Sending 6 messages to LLM...
--- Node 'llm' ---
  ↳ Planning to call tool: tavily_search
[🛠️] Calling tool: tavily_search with args: {'query': 'current weather in São Paulo, Brazil'}
[✅] Returning to LLM with tool results.
--- Node 'action' ---
{'query': 'current weather in São Paulo, Brazil', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in São Paulo, Brazil', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Sao Paulo', 'region': 'Sao Paulo', 'country': 'Brazil', 'lat': -23.5333, 'lon': -46.6167, 'tz_id': 'America/Sao_Paulo', 'localtime_epoch': 1769283985, 'localtime': '2026-01-24 16:46'}, 'current': {'last_updated_epoch': 1769283900, 'last_updated': '2026-01-24 16:45', 'temp_c': 22.2, 'temp_f': 72.0, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weather

In [7]:
# 8. Test 2 - Agent Initialization (Advanced Prompt & Compatible Tool Naming)

from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage
from datetime import datetime
from langchain_core.tools import tool

# --- 1. Tool Renaming (Compatible Syntax) ---
# Em vez de usar o argumento 'name', definimos a função com o nome desejado.
# A descrição (essencial para o LLM) é a docstring da função.
@tool
def tavily_search_results_json(query: str) -> str:
    """
    An intelligent search engine. Use this to find real-time information on any topic.
    The input should be a precise search query.
    Returns a JSON object with search results.
    """
    search_tool = TavilySearch(max_results=3)
    return search_tool.invoke(query)

print(f"[✅] Tool configured with declarative name: '{tavily_search_results_json.name}'")

# --- 2. Advanced System Prompt ---
current_date = datetime.now().strftime("%Y-%m-%d")

# A f-string agora pega o nome da nova função
tool_name = tavily_search_results_json.name 

SYSTEM_PROMPT = f"""
You are an intelligent research assistant. Today's date is {current_date}.

Core Instruction: Use the search engine ({tool_name}) to look for information.

Behavioral Rules:
- You are allowed to make multiple tool calls.
- Only search when you are sure what to look for.
- When asked to compare information, use information from the conversation history and tool results.
- CRITICAL: After using the tool and getting results, you MUST provide a final answer. Do not loop.
""".strip()
print("[✅] Advanced system prompt defined.")

# --- 3. Agent Initialization ---
# Passamos a função com o nome novo
abot = Agent(
    llm,
    [tavily_search_results_json], # ### MUDANÇA ###
    system=SYSTEM_PROMPT,
    checkpointer=memory
)
print("[✅] Agent initialized with advanced prompt.")

# --- 4. Execution ---
thread_config = {"configurable": {"thread_id": "conversation-6"}} # Novo ID

messages = [HumanMessage(content="What is the current weather in São Paulo, Brazil?")]

print("\n" + "="*70)
print(f"🚀 THREAD: {thread_config['configurable']['thread_id']}")
print("="*70)

for event in abot.graph.stream({"messages": messages}, thread_config):
    for node_name, value in event.items():
        print(f"--- Node '{node_name}' ---")
        last_message = value['messages'][-1]
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print(f"  ↳ Planning to call tool: {last_message.tool_calls[0]['name']}")
        else:
            print(last_message.content)

[✅] Tool configured with declarative name: 'tavily_search_results_json'
[✅] Advanced system prompt defined.
[✅] Agent initialized with advanced prompt.

🚀 THREAD: conversation-6
[🧠] Sending 3 messages to LLM...
--- Node 'llm' ---
  ↳ Planning to call tool: tavily_search_results_json
[🛠️] Calling tool: tavily_search_results_json with args: {'query': 'current weather in Sao Paulo, Brazil'}
[✅] Returning to LLM with tool results.
--- Node 'action' ---
{'query': 'current weather in Sao Paulo, Brazil', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in Sao Paulo, Brazil', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Sao Paulo', 'region': 'Sao Paulo', 'country': 'Brazil', 'lat': -23.5333, 'lon': -46.6167, 'tz_id': 'America/Sao_Paulo', 'localtime_epoch': 1769284032, 'localtime': '2026-01-24 16:47'}, 'current': {'last_updated_epoch': 1769283900, 'last_updated': '2026-01-24 16:45', 'temp_c': 22.2, 'temp_f': 72.0, 'is_day': 1,

In [8]:
# 7. Memory & Streaming Test (Final Exam)

from langchain_core.messages import HumanMessage

print("="*70)
print("🧠 MEMORY & STREAMING TEST")
print("="*70)
print("This test will verify if the agent can remember previous turns in a conversation.")

# --- Test 1: Follow-up question in the SAME thread ---
# We use the SAME thread_id as the previous cell ("conversation-6")
# The agent should remember we were talking about weather.
messages_rio = [HumanMessage(content="And in Rio de Janeiro?")]
thread_config = {"configurable": {"thread_id": "conversation-6"}}

print("\n--- 💬 Test 1: Follow-up in Thread 'conversation-6' (Rio de Janeiro) ---")
for event in abot.graph.stream({"messages": messages_rio}, thread_config):
    for node_name, value in event.items():
        print(f"--- Node '{node_name}' ---")
        last_message = value['messages'][-1]
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print(f"  ↳ Planning to call tool: {last_message.tool_calls[0]['name']}")
        else:
            print(last_message.content)

# --- Test 2: Comparison question in the SAME thread ---
# The agent should use its memory of BOTH São Paulo and Rio to answer.
# EXPECTED: No tool call needed.
messages_compare = [HumanMessage(content="Which one was hotter?")]
# Still using the same thread
# thread_config = {"configurable": {"thread_id": "conversation-6"}} 

print("\n--- 🤔 Test 2: Comparison with Memory (Thread 'conversation-6') ---")
for event in abot.graph.stream({"messages": messages_compare}, thread_config):
    for node_name, value in event.items():
        print(f"--- Node '{node_name}' ---")
        last_message = value['messages'][-1]
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print(f"  ↳ Planning to call tool: {last_message.tool_calls[0]['name']} (Note: Should not happen if memory works)")
        else:
            print(last_message.content)

# --- Test 3: Comparison question in a NEW thread ---
# The agent has NO memory of São Paulo or Rio in this new thread.
# EXPECTED: It should be confused, ask for clarification, or make a new search.
messages_new_thread = [HumanMessage(content="Which one is hotter?")]
thread_config_new = {"configurable": {"thread_id": "conversation-7"}} # A brand new thread

print("\n--- 🤷 Test 3: Comparison without Memory (NEW Thread 'conversation-7') ---")
for event in abot.graph.stream({"messages": messages_new_thread}, thread_config_new):
    for node_name, value in event.items():
        print(f"--- Node '{node_name}' ---")
        last_message = value['messages'][-1]
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print(f"  ↳ Planning to call tool: {last_message.tool_calls[0]['name']} (As expected, it needs to search again)")
        else:
            print(last_message.content)

print("\n" + "="*70)
print("[🎉] Memory and Streaming tests completed!")
print("="*70)

🧠 MEMORY & STREAMING TEST
This test will verify if the agent can remember previous turns in a conversation.

--- 💬 Test 1: Follow-up in Thread 'conversation-6' (Rio de Janeiro) ---
[🧠] Sending 7 messages to LLM...
--- Node 'llm' ---
  ↳ Planning to call tool: tavily_search_results_json
[🛠️] Calling tool: tavily_search_results_json with args: {'query': 'current weather in Rio de Janeiro, Brazil'}
[✅] Returning to LLM with tool results.
--- Node 'action' ---
{'query': 'current weather in Rio de Janeiro, Brazil', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in Rio de Janeiro, Brazil', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Rio De Janeiro', 'region': 'Rio de Janeiro', 'country': 'Brazil', 'lat': -22.9, 'lon': -43.2333, 'tz_id': 'America/Sao_Paulo', 'localtime_epoch': 1769284431, 'localtime': '2026-01-24 16:53'}, 'current': {'last_updated_epoch': 1769283900, 'last_updated': '2026-01-24 16:45', 'temp_c': 25.0, 'te